# Semantic search over documentation with LanceDB

This notebook builds a searchable index of the [FastAPI documentation](https://fastapi.tiangolo.com) with [LanceDB](https://docs.lancedb.com). It downloads the docs, splits each page into sections, stores the sections in a LanceDB table that embeds text automatically, and answers natural-language questions with vector search.

**Before you start**

- Python 3.10 or later (tested with 3.12)
- Git, used to download the docs
- The packages in [`requirements.txt`](requirements.txt), installed in a virtual environment
- About 500 MB of free disk space for the embedding model, PyTorch, and the docs
- No API keys: embeddings run locally with the `all-MiniLM-L6-v2` model

The first run downloads the FastAPI docs from GitHub and the embedding model from Hugging Face, so it needs an internet connection. Later runs reuse both.

**Data source and license:** The FastAPI documentation is © Sebastián Ramírez and contributors, used under the [MIT License](https://github.com/fastapi/fastapi/blob/master/LICENSE). The notebook downloads it when it runs; this repository doesn't include a copy.

## Setup

Settings used throughout the notebook. `FASTAPI_REF` pins the docs to one FastAPI release, so results are reproducible.

In [ ]:
import re
import subprocess
from pathlib import Path

import lancedb
import pandas as pd
from lancedb.embeddings import get_registry
from lancedb.pydantic import LanceModel, Vector

FASTAPI_REF = "0.159.0"          # FastAPI release whose docs we index
DATA_DIR = Path("data")          # Everything this notebook downloads or creates
DOCS_REPO = DATA_DIR / "fastapi"
DOCS_ROOT = DOCS_REPO / "docs" / "en" / "docs"
DB_PATH = DATA_DIR / "lancedb"
TABLE_NAME = "fastapi_docs"
MODEL_NAME = "all-MiniLM-L6-v2"

print("LanceDB version:", lancedb.__version__)

## Step 1: Download the docs

A sparse, shallow Git clone fetches only the English docs folder at the pinned release, instead of the whole repository and its history.

In [ ]:
def download_docs(ref: str = FASTAPI_REF) -> None:
    if DOCS_ROOT.exists():
        print(f"Docs already downloaded to {DOCS_ROOT}")
        return
    DATA_DIR.mkdir(exist_ok=True)
    subprocess.run(
        ["git", "clone", "--quiet", "--depth", "1", "--branch", ref,
         "--filter=blob:none", "--sparse",
         "https://github.com/fastapi/fastapi.git", str(DOCS_REPO)],
        check=True,
    )
    subprocess.run(["git", "-C", str(DOCS_REPO), "sparse-checkout", "set", "docs/en/docs"], check=True)
    print(f"Downloaded FastAPI {ref} docs to {DOCS_ROOT}")

download_docs()
pages = sorted(DOCS_ROOT.rglob("*.md"))
print(f"{len(pages)} Markdown pages found")

## Step 2: Clean and split the pages

Search works better on focused passages than on whole pages, so each page is split at its `##` headings. Very long sections are split again at paragraph breaks, because the embedding model reads only about the first 250 words of a passage.

The cleaning step removes markup that only means something to the docs site's build tool: heading anchors such as `{ #query-parameters }`, code-include directives such as `{* ../../docs_src/... *}`, admonition fences (`///`), and HTML tags. Pages with little prose to search, such as the release notes and the generated API reference stubs, are skipped.

In [ ]:
SKIP_PAGES = {
    "release-notes.md", "_llm-test.md", "translation-banner.md",
    "fastapi-people.md", "management.md", "management-tasks.md", "newsletter.md",
}
SKIP_FOLDERS = {"reference"}      # Generated API reference stubs
MAX_WORDS = 250                   # Roughly what the embedding model reads

DIRECTIVE = re.compile(r"^\s*(\{\*.*\*\}|\{!.*!\}|:::|////?)")
ANCHOR = re.compile(r"\s*\{\s*#[^}]*\}\s*$")
HTML_TAG = re.compile(r"<[^>]+>")


def clean_lines(text: str) -> list[str]:
    lines = []
    for line in text.splitlines():
        if DIRECTIVE.match(line):
            continue
        line = ANCHOR.sub("", line)
        line = HTML_TAG.sub("", line)
        lines.append(line)
    return lines


def page_url(path: Path) -> str:
    rel = path.relative_to(DOCS_ROOT).with_suffix("")
    parts = [] if rel.name == "index" and rel.parent == Path(".") else list(rel.parts)
    if parts and parts[-1] == "index":
        parts = parts[:-1]
    return "https://fastapi.tiangolo.com/" + "".join(p + "/" for p in parts)


def split_long(text: str, max_words: int = MAX_WORDS) -> list[str]:
    if len(text.split()) <= max_words:
        return [text]
    parts, current = [], []
    for para in re.split(r"\n\s*\n", text):
        if current and len(" ".join(current + [para]).split()) > max_words:
            parts.append("\n\n".join(current))
            current = []
        current.append(para)
    if current:
        parts.append("\n\n".join(current))
    return parts


def chunk_page(path: Path) -> list[dict]:
    rel = path.relative_to(DOCS_ROOT)
    section = rel.parts[0] if len(rel.parts) > 1 else "general"
    title, heading, body, in_code = path.stem, "Introduction", [], False
    sections = []
    for line in clean_lines(path.read_text(encoding="utf-8")):
        if line.lstrip().startswith("```"):
            in_code = not in_code
        if not in_code and line.startswith("# "):
            title = line[2:].strip()
            continue
        if not in_code and line.startswith("## "):
            sections.append((heading, body))
            heading, body = line[3:].strip(), []
            continue
        body.append(line)
    sections.append((heading, body))

    chunks = []
    for heading, lines in sections:
        text = "\n".join(lines).strip()
        if len(text.split()) < 15:            # Skip empty or near-empty sections
            continue
        for i, part in enumerate(split_long(text)):
            chunks.append({
                "id": f"{rel.as_posix()}#{len(chunks)}",
                "page_title": title,
                "heading": heading,
                "section": section,
                "url": page_url(path),
                "word_count": len(part.split()),
                "text": f"{title}: {heading}\n\n{part}",
            })
    return chunks


chunks = []
for path in pages:
    rel = path.relative_to(DOCS_ROOT)
    if path.name in SKIP_PAGES or rel.parts[0] in SKIP_FOLDERS:
        continue
    chunks.extend(chunk_page(path))

df = pd.DataFrame(chunks)
print(f"{len(df)} chunks from {df['url'].nunique()} pages")
df.groupby("section").size().sort_values(ascending=False)

In [ ]:
df[["page_title", "heading", "section", "word_count", "url"]].head(10)

## Step 3: Create the table with automatic embeddings

LanceDB's embedding registry ties an embedding model to the table's schema. The `text` field is marked as the source and `vector` as the destination, so LanceDB embeds text automatically when rows are added and when you search with a string. You never call the model yourself.

The first time this cell runs, it downloads the model (about 90 MB). Embedding all the chunks takes a minute or two on a laptop CPU.

In [ ]:
embedder = get_registry().get("sentence-transformers").create(name=MODEL_NAME, device="cpu")


class DocChunk(LanceModel):
    id: str
    page_title: str
    heading: str
    section: str
    url: str
    word_count: int
    text: str = embedder.SourceField()
    vector: Vector(embedder.ndims()) = embedder.VectorField()


db = lancedb.connect(DB_PATH)
table = db.create_table(TABLE_NAME, schema=DocChunk, mode="overwrite")
table.add(df.to_dict("records"))
print(f"Table '{TABLE_NAME}' has {table.count_rows()} rows; vectors have {embedder.ndims()} dimensions")

## Step 4: Ask questions with vector search

Vector search compares the meaning of the question with the meaning of each chunk, so it finds relevant passages even when they use different words. A smaller `_distance` means a closer match.

In [ ]:
def ask(question: str, k: int = 5) -> pd.DataFrame:
    return (
        table.search(question)
        .limit(k)
        .select(["page_title", "heading", "url", "_distance"])
        .to_pandas()
    )


ask("How do I make a query parameter optional?")

In [ ]:
ask("How can I run code after returning a response?")

## Next steps

The next sections of this notebook will add filtering by docs section, full-text and hybrid search, a vector index, and table versioning.